<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/03b-transfer-learning-and-unsupervised-pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash

pip install torchinfo torchmetrics

In [ ]:
import copy


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import decomposition, metrics, model_selection, pipeline, preprocessing
import torch
from torch import nn, optim, utils
import torchinfo
import torchmetrics


### Define some utility functions

The code in the cell below defines a few utility functions that will make our life easier.

In [ ]:
def compute_average_loss_per_batch(dataloader, criterion, model_fn):
    total_loss = torch.zeros(1, 1)
    num_batches = len(dataloader)
    for features, targets in dataloader:
        predictions = model_fn(features)
        batch_loss = criterion(predictions, targets)
        total_loss += batch_loss
    average_loss_per_batch = total_loss / num_batches
    return average_loss_per_batch


def clip_gradients_(
    clip_grad_strategy,
    model_fn,
    clip_value=None,
    error_if_nonfinite=False,
    max_norm=None,
    norm_type=2.0):
    if clip_grad_strategy == "value" and clip_value is not None:
        nn.utils.clip_grad_value_(
            model_fn.parameters(),
            clip_value
        )
    elif clip_grad_strategy == "norm" and max_norm is not None:
        nn.utils.clip_grad_norm_(
            model_fn.parameters(),
            max_norm,
            norm_type,
            error_if_nonfinite
        )
    else:
        raise NotImplementedError()


def evaluate(model_fn, dataloader, metric, device="cpu"):
    model_fn.eval()
    metric.reset()
    with torch.inference_mode():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model_fn(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device="cpu",
    clip_grad_strategy=None,
    clip_value=None,
    error_if_nonfinite=False,
    log_epochs=1,
    max_epochs=1,
    max_norm=None,
    norm_type=2.0
    ):

    history = {
        "epoch": [],
        "average_train_loss": [],
        "average_val_loss": []
    }
    num_train_batches = len(train_dataloader)
    for epoch in range(max_epochs):
        total_train_loss = torch.zeros(1, 1)
        model_fn = model_fn.train()
        for features, targets in train_dataloader:

            # forward pass
            features, targets = features.to(device), targets.to(device)
            predictions = model_fn(features)
            batch_loss = criterion(predictions, targets)
            total_train_loss += batch_loss

            # backward pass
            optimizer.zero_grad()
            batch_loss.backward()
            if clip_grad_strategy is not None:
                clip_gradients_(
                    clip_grad_strategy,
                    model_fn,
                    clip_value,
                    error_if_nonfinite,
                    max_norm,
                    norm_type
                )
            optimizer.step()

        history["epoch"].append(epoch)

        average_train_loss_per_batch = total_train_loss / num_train_batches
        history["average_train_loss"].append(average_train_loss_per_batch.item())

        model_fn = model_fn.eval()
        with torch.inference_mode():
            average_val_loss_per_batch = compute_average_loss_per_batch(
                val_dataloader,
                criterion,
                model_fn
            )
        history["average_val_loss"].append(average_val_loss_per_batch.item())


        if (epoch + 1) % log_epochs == 0:
            print(
                f"Epoch {epoch},",
                f"Average train Loss {average_train_loss_per_batch.item():.4f},",
                f"Average val Loss {average_val_loss_per_batch.item():.4f}"
            )

    history_df = (
        pd.DataFrame.from_dict(history)
                    .set_index("epoch")
    )

    return history_df


In [ ]:
def initialize_linear_layer(
    in_features,
    out_features,
    init_strategy_=nn.init.kaiming_uniform_,
    init_strategy_kwargs=None,
    ):
    linear_layer = nn.Linear(in_features, out_features)

    if init_strategy_ is not None:
        if init_strategy_kwargs is None:
            init_strategy_kwargs = {}
        init_strategy_(linear_layer.weight, **init_strategy_kwargs)
        linear_layer.bias.data.fill_(0.0)

    return linear_layer


def make_mlp_classifier(
    input_size,
    hidden_sizes=None,
    output_size=2,
    activation_fn=None,
    init_strategy_=nn.init.kaiming_uniform_,
    init_strategy_kwargs=None,
    batch_normalization=False
    ):
    modules = []
    hidden_sizes = [] if hidden_sizes is None else hidden_sizes
    for hidden_size in hidden_sizes:
        hidden_layer = initialize_linear_layer(
            input_size,
            hidden_size,
            init_strategy_,
            init_strategy_kwargs,
        )
        modules.append(hidden_layer)

        # batch normalization goes after the linear layer...
        if batch_normalization:
            modules.append(nn.BatchNorm1d(hidden_size))

        # ...but before the activation_fn!
        if activation_fn is not None:
            modules.append(activation_fn)
        input_size=hidden_size
    output_layer = initialize_linear_layer(
            input_size,
            output_size,
            init_strategy_,
            init_strategy_kwargs,
    )
    modules.append(output_layer)
    model_fn = nn.Sequential(*modules)
    return nn.CrossEntropyLoss(), model_fn


# Transfer Learning

In this section we will train a DNN model on the MNIST dataset and then use this pre-trained model as a starting point for training a model to classify an Arabic version of the MNIST dataset.

## Load the MNIST data

In [ ]:
INPUT_SIZE = 784
OUTPUT_SIZE = 10
RANDOM_STATE = np.random.RandomState(42)


_train_data_df = pd.read_csv(
    "./sample_data/mnist_train_small.csv",
    header=None,
    names=["label"] + [f"p{i}" for i in range(INPUT_SIZE)],
)
train_data_df, val_data_df = model_selection.train_test_split(
    _train_data_df,
    random_state=RANDOM_STATE,
    stratify=_train_data_df.loc[:, "label"],
    test_size=0.1,
)

test_data_df = pd.read_csv(
    "./sample_data/mnist_test.csv",
    header=None,
    names=["label"] + [f"p{i}" for i in range(INPUT_SIZE)],
)

### Create preprocessing pipelines

In [ ]:
def array_to_tensor(arr, dtype=torch.float32):
  return torch.tensor(arr, dtype=dtype)


def series_to_tensor(s, dtype=torch.float32):
    arr = s.to_numpy()
    return array_to_tensor(arr, dtype)


features_preprocessor = pipeline.make_pipeline(
    preprocessing.StandardScaler(),
    preprocessing.FunctionTransformer(
        array_to_tensor,
        kw_args={
            "dtype":
            torch.float32
        }
    ),
)

target_preprocessor = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        series_to_tensor,
        kw_args={
            "dtype":
            torch.int64
        }
    ),
)


### Create Datasets and DataLoaders

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2


# create the training dataset and dataloader
train_features_tensor = features_preprocessor.fit_transform(
    train_data_df.drop("label", axis=1)
)

train_target_tensor = target_preprocessor.fit_transform(
    train_data_df.loc[:, "label"]
)

train_dataset = utils.data.TensorDataset(
    train_features_tensor,
    train_target_tensor
)

mnist_train_dataloader = utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    persistent_workers=True,
    num_workers=NUM_WORKERS,
    shuffle=True,
)

# create the validation dataset and dataloader
val_features_tensor = features_preprocessor.transform(
    val_data_df.drop("label", axis=1)
)

val_target_tensor = target_preprocessor.transform(
    val_data_df.loc[:, "label"]
)

val_dataset = utils.data.TensorDataset(
    val_features_tensor,
    val_target_tensor
)

mnist_val_dataloader = utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    persistent_workers=True,
    num_workers=NUM_WORKERS,
    shuffle=False
)

# create the test dataset and dataloader
test_features_tensor = features_preprocessor.transform(
    test_data_df.drop("label", axis=1)
)

test_target_tensor = target_preprocessor.transform(
    test_data_df.loc[:, "label"]
)

test_dataset = utils.data.TensorDataset(
    test_features_tensor,
    test_target_tensor
)

mnist_test_dataloader = utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    persistent_workers=False,
    num_workers=NUM_WORKERS,
    shuffle=False
)


## Train a DNN on the MNIST data

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HIDDEN_SIZE = int((2 / 3) * (INPUT_SIZE + OUTPUT_SIZE))
LEARNING_RATE = 1e-3
MAX_EPOCHS = 50


criterion, pretrained_mnist_model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SELU(),
    init_strategy_=nn.init.kaiming_normal_,
    init_strategy_kwargs={
        "mode": "fan_in",
        "nonlinearity": "linear"
    }
)

optimizer = optim.SGD(
    pretrained_mnist_model_fn.parameters(),
    lr=LEARNING_RATE
)

mnist_training_history_df = fit(
    criterion,
    pretrained_mnist_model_fn,
    optimizer,
    mnist_train_dataloader,
    mnist_val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = mnist_training_history_df.plot(grid=True)

## Download Arabic Handwritten Digits Data

Here we will download the Arabic MNIST dataset. All images have the same size 28x28 = 784 pixels as the orginal MNIST data; there are also the same number of classes in this dataset.

In [ ]:
%%bash

gdown 1_aWjSUmpBlLSTSJF4xhmdZnzMoK4AMRO
gdown 1Syln7zCy9Ue8x_F5qgq_0EHvaL_h_oL5
gdown 1QQUsu7QciAZ6KXt6u-ZaMT_61iNjgy0a

Let's pretend that we only have 10k labeled images and that the remaining 60k images are all unlabeled.

In [ ]:
_test_features_df = pd.read_csv(
    "/content/csvTestImages 10k x 784.csv",
    header=None,
    names=[f"p{i}" for i in range(INPUT_SIZE)],
)

_test_target = (
    pd.read_csv(
        "/content/csvTestLabel 10k x 1.csv",
        header=None,
        names=["label"],
    ).loc[:, "label"]
)

arabic_mnist_train_features_df, arabic_mnist_val_features_df, arabic_mnist_train_target, arabic_mnist_val_target = (
    model_selection.train_test_split(
      _test_features_df,
      _test_target,
      random_state=RANDOM_STATE,
      shuffle=True,
      stratify=_test_target,
      test_size=0.1,
    )
)


In [ ]:
# create the training dataset and dataloader
_train_features_tensor = features_preprocessor.fit_transform(
    arabic_mnist_train_features_df
)

_train_target_tensor = target_preprocessor.fit_transform(
    arabic_mnist_train_target
)

_train_dataset = utils.data.TensorDataset(
    _train_features_tensor,
    _train_target_tensor
)

arabic_mnist_train_dataloader = utils.data.DataLoader(
    _train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    persistent_workers=True,
    shuffle=True,
)

# create the validation dataset and dataloader
_val_features_tensor = features_preprocessor.transform(
    arabic_mnist_val_features_df
)

_val_target_tensor = target_preprocessor.transform(
    arabic_mnist_val_target
)

_val_dataset = utils.data.TensorDataset(
    _val_features_tensor,
    _val_target_tensor
)

arabic_mnist_val_dataloader = utils.data.DataLoader(
    _val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False,
)

### Logistic Regression Benchmark

First, we can use logistic regression as a benchmark model. Our more elaborate models should seek to outperform this benchmark.

In [ ]:
criterion, benchmark_model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    output_size=OUTPUT_SIZE,
)

optimizer = optim.SGD(
    benchmark_model_fn.parameters(),
    lr=LEARNING_RATE
)

logistic_regression_history_df = fit(
    criterion,
    benchmark_model_fn,
    optimizer,
    arabic_mnist_train_dataloader,
    arabic_mnist_val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = logistic_regression_history_df.plot(grid=True)

### Freeze the pre-trained backbone and fine-tune the classifier head

In [ ]:
torchinfo.summary(pretrained_mnist_model_fn)

In [ ]:
classifier_only_fine_tune_model_fn = copy.deepcopy(pretrained_mnist_model_fn)
classifier_only_fine_tune_model_fn = classifier_only_fine_tune_model_fn.to(DEVICE)

In [ ]:
torchinfo.summary(classifier_only_fine_tune_model_fn)

In [ ]:
# freeze layers 0 through 5 (inclusive!)
for i in range(0, 5 + 1):
    for p in classifier_only_fine_tune_model_fn[i].parameters():
        p.requires_grad = False

In [ ]:
torchinfo.summary(classifier_only_fine_tune_model_fn)

In [ ]:
optimizer = optim.SGD(
    classifier_only_fine_tune_model_fn.parameters(),
    lr=LEARNING_RATE
)

classifier_only_fine_tune_history_df = fit(
    criterion,
    classifier_only_fine_tune_model_fn,
    optimizer,
    arabic_mnist_train_dataloader,
    arabic_mnist_val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = classifier_only_fine_tune_history_df.plot(grid=True)

### Unfreezing deeper layers of the backbone

Just fine-tuning the classifier head didn't seem to work very well. Lets try to un-freeze additional layers.

In [ ]:
partial_fine_tune_model_fn = copy.deepcopy(pretrained_mnist_model_fn)
partial_fine_tune_model_fn = partial_fine_tune_model_fn.to(DEVICE)

# freeze layers 0 through 3 (inclusive!)
for i in range(0, 3 + 1):
    for p in partial_fine_tune_model_fn[i].parameters():
        p.requires_grad = False

In [ ]:
torchinfo.summary(partial_fine_tune_model_fn)

In [ ]:
optimizer = optim.SGD(
    partial_fine_tune_model_fn.parameters(),
    lr=LEARNING_RATE
)

partial_fine_tune_history_df = fit(
    criterion,
    partial_fine_tune_model_fn,
    optimizer,
    arabic_mnist_train_dataloader,
    arabic_mnist_val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = partial_fine_tune_history_df.plot(grid=True)

Now we are seeing some improvement!

### Full fine-tune of the pre-trained model

You can also try unfreezing all of the layers and fine-tune the pre-trained model on your new data.

In [ ]:
full_fine_tune_model_fn = copy.deepcopy(pretrained_mnist_model_fn)
full_fine_tune_model_fn = full_fine_tune_model_fn.to(DEVICE)

In [ ]:
optimizer = optim.SGD(
    full_fine_tune_model_fn.parameters(),
    lr=LEARNING_RATE
)

full_fine_tune_history_df = fit(
    criterion,
    full_fine_tune_model_fn,
    optimizer,
    arabic_mnist_train_dataloader,
    arabic_mnist_val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = full_fine_tune_history_df.plot(grid=True)

### Train model from scratch

In [ ]:
criterion, from_scratch_model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SELU(),
    init_strategy_=nn.init.kaiming_normal_,
    init_strategy_kwargs={
        "mode": "fan_in",
        "nonlinearity": "linear"
    }
)
from_scratch_model_fn = from_scratch_model_fn.to(DEVICE)

optimizer = optim.SGD(
    from_scratch_model_fn.parameters(),
    lr=LEARNING_RATE
)

from_scratch_history_df = fit(
    criterion,
    from_scratch_model_fn,
    optimizer,
    arabic_mnist_train_dataloader,
    arabic_mnist_val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 5), sharey=True)

_ = classifier_only_fine_tune_history_df.plot(ax=axes[0], grid=True, title="Fine-tune Classifier Only")
_ = partial_fine_tune_history_df.plot(ax=axes[1], grid=True, title="Partial Fine-tune")
_ = full_fine_tune_history_df.plot(ax=axes[2], grid=True, title="Full Fine-tune")
_ = from_scratch_history_df.plot(ax=axes[3], grid=True, title="Train from Scratch")

### Exercise:

Compare the performance of the original MNIST model at predicting the Arabic Handwritten digits, the performance of the Arabic MNIST model that was fine-tuned on the Arabic MNIST data, and the Arabic MNIST model that was trained from scratch.

#### Solution:

In [ ]:
accuracy_metric = (
    torchmetrics.Accuracy(
        task="multiclass",
        num_classes=OUTPUT_SIZE,
    ).to(DEVICE)
)

pretrained_mnist_model_accuracy = evaluate(
    pretrained_mnist_model_fn,
    arabic_mnist_val_dataloader,
    accuracy_metric,
    device=DEVICE,
)

full_fine_tune_accuracy = evaluate(
    full_fine_tune_model_fn,
    arabic_mnist_val_dataloader,
    accuracy_metric,
    device=DEVICE,
)

from_scratch_accuracy = evaluate(
    from_scratch_model_fn,
    arabic_mnist_val_dataloader,
    accuracy_metric,
    device=DEVICE,
)

print(f"Pretrained MNIST model accuracy: {pretrained_mnist_model_accuracy.item(): .4f}")
print(f"Full fine-tuned Arabic MNIST model accuracy: {full_fine_tune_accuracy.item(): .4f}")
print(f"From scratch Arabic MNIST model accuracy: {from_scratch_accuracy.item(): .4f}")

# Unsupervised pre-training

Often you will have a large amount of unlabeled data and a small amount of labeled data. In this situation, one possible solution is to use unsupervised pre-training.

In [ ]:
arabic_mnist_unlabeled_features_df = pd.read_csv(
    "/content/csvTrainImages 60k x 784.csv",
    header=None,
    names=[f"p{i}" for i in range(INPUT_SIZE)],
)


## Generate an embedding

When applying unsupervised pre-training you first need to find a "good" embedding of your unlabeled features. For computer vision applications you would want to use more powerful models such as autoencoders, generative adversarial models (GANs) or similar to generate your embeddings. Here we just use PCA.

In [ ]:
encoder_pipeline = pipeline.make_pipeline(
    preprocessing.StandardScaler(),
    decomposition.PCA(
        n_components=0.95,
        whiten=True,
    ),
)

In [ ]:
feature_embedding = encoder_pipeline.fit_transform(
    arabic_mnist_unlabeled_features_df
)

In [ ]:
_, input_size = feature_embedding.shape

In [ ]:
print(input_size)

How do we define a "good" embedding? Reconstruction error! Once we have a feature embedding, we can invert the embedding transformation in order to reconstruct the original features. Once we have a reconstruction of the original features we can estimate the reconstruction error by comparing the original features and the reconstructed features.

A good embedding will have a small reconstruction error.

In [ ]:
reconstructed_features = encoder_pipeline.inverse_transform(feature_embedding)
metrics.root_mean_squared_error(
    arabic_mnist_unlabeled_features_df.to_numpy(),
    reconstructed_features
)

### Exercise:

How could we improve the performance of our feature encoder pipeline? What impact will improving the performance of the feature encoder pipeline have on the performance of the DNN trained on the encoded features?

#### Solution:

Simple way to improve the PCA encoder pipeline above would be to increase the number of components!

Alternatively you could try a non-linear encoder pipeline using Locally-Linear Embedding. This pipeline consists of two steps:

1.  **`preprocessing.StandardScaler()`**: This step standardizes features by removing the mean and scaling to unit variance. This is important for many machine learning algorithms, including LLE, as it helps prevent features with larger values from dominating the distance calculations.

2.  **`manifold.LocallyLinearEmbedding(...)`**: This is the core LLE component. Locally Linear Embedding is a non-linear dimensionality reduction technique. It attempts to find a low-dimensional projection of the data that preserves the local neighborhood structure of the data points.
    *   `n_components`: Specifies the desired dimensionality of the output embedding. Here, it's set to 2, which is suitable for visualization.
    *   `n_neighbors`: Defines the number of nearest neighbors to consider for each point when constructing the local models. This parameter is crucial for LLE's performance.
    *   `random_state`: Ensures reproducibility of the results.
    *   `n_jobs`: Uses all available CPU cores for parallel processing, speeding up the computation.

In [ ]:
from sklearn import manifold


encoder_pipeline = pipeline.make_pipeline(
    preprocessing.StandardScaler(),
    manifold.LocallyLinearEmbedding(
        n_components=2,
        n_neighbors=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
)
non_linear_feature_embedding = encoder_pipeline.fit_transform(
    arabic_mnist_unlabeled_features_df
)


reconstructed_features = encoder_pipeline.inverse_transform(
    non_linear_feature_embedding
)
lle_rmse = metrics.root_mean_squared_error(
    arabic_mnist_unlabeled_features_df.to_numpy(),
    reconstructed_features
)

print(f"Reconstruction error of LLE: {lle_rmse: .4f}")

## Use encoder pipeline to embed your labeled data

Hopefully our "good" embedding of the unlabeled features has learned useful information for our supervised classification task. Now we use the trained encoder pipeline to embed our labeled features.

In [ ]:
embedded_train_features = encoder_pipeline.transform(
    arabic_mnist_train_features_df
)
embedded_val_features = encoder_pipeline.transform(
    arabic_mnist_val_features_df
)


Now we need to prepare out data for our DNN.

In [ ]:
features_preprocessor = pipeline.make_pipeline(
    preprocessing.StandardScaler(),
    preprocessing.FunctionTransformer(
        array_to_tensor,
        kw_args={
            "dtype": torch.float32,
        }
    ),
)

target_preprocessor = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        series_to_tensor,
        kw_args={
            "dtype": torch.int64,
        }
    )
)

# create the training dataset and dataloader
_train_features_tensor = features_preprocessor.fit_transform(
    embedded_train_features
)

_train_target_tensor = target_preprocessor.fit_transform(
    arabic_mnist_train_target
)

_train_dataset = utils.data.TensorDataset(
    _train_features_tensor,
    _train_target_tensor
)

embedded_train_dataloader = utils.data.DataLoader(
    _train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    persistent_workers=True,
    shuffle=True,
)

# create the validation dataset and dataloader
_val_features_tensor = features_preprocessor.transform(
    embedded_val_features
)

_val_target_tensor = target_preprocessor.transform(
    arabic_mnist_val_target
)

_val_dataset = utils.data.TensorDataset(
    _val_features_tensor,
    _val_target_tensor
)

embedded_val_dataloader = utils.data.DataLoader(
    _val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False,
)

### Train a DNN using the embedded features

In [ ]:
hidden_size = int((2 / 3) * (input_size + OUTPUT_SIZE))
criterion, unsupervised_pretraining_model_fn = make_mlp_classifier(
    input_size,
    hidden_sizes=[hidden_size, hidden_size, hidden_size],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SELU(),
    init_strategy_=nn.init.kaiming_normal_,
    init_strategy_kwargs={
        "mode": "fan_in",
        "nonlinearity": "linear"
    }
)
unsupervised_pretraining_model_fn = unsupervised_pretraining_model_fn.to(DEVICE)

optimizer = optim.SGD(
    unsupervised_pretraining_model_fn.parameters(),
    lr=LEARNING_RATE
)

unsupervised_pretraining_history_df = fit(
    criterion,
    unsupervised_pretraining_model_fn,
    optimizer,
    embedded_train_dataloader,
    embedded_val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

### Exercise:

Compare the performance of the transfer learning and unsupervised pre-training approaches to classifying the Arabic Handwritten Digits images.

#### Solution:

In [ ]:
accuracy_metric = (
    torchmetrics.Accuracy(
        task="multiclass",
        num_classes=OUTPUT_SIZE,
    ).to(DEVICE)
)

full_fine_tune_accuracy = evaluate(
    full_fine_tune_model_fn,
    arabic_mnist_val_dataloader,
    accuracy_metric,
    device=DEVICE,
)

unsupervised_pretraining_accuracy = evaluate(
    unsupervised_pretraining_model_fn,
    embedded_val_dataloader,
    accuracy_metric,
    device=DEVICE,
)

print(f"Transfer learning model accuracy: {full_fine_tune_accuracy.item(): .4f}")
print(f"Unsupervised pretraining model accuracy: {unsupervised_pretraining_accuracy.item(): .4f}")

### Exercise:

How could you combine both transfer learning and unsupervised pre-training? Combine transfer learning and unsupervised pre-training and see if it improves the task performance on the validation data.

#### Solution:

In [ ]:
# encoder output dimension needs to match the pretrained model!
encoder_pipeline = pipeline.make_pipeline(
    preprocessing.StandardScaler(),
    decomposition.PCA(
        n_components=None,  # computes all 784 components!
        whiten=True,
        svd_solver="full",
    ),
)

_ = encoder_pipeline.fit(
    arabic_mnist_unlabeled_features_df
)

# embedded the label training and validation data
_embedded_train_features = encoder_pipeline.transform(
    arabic_mnist_train_features_df
)
_embedded_val_features = encoder_pipeline.transform(
    arabic_mnist_val_features_df
)

# create the training dataset and dataloader
_embedded_train_features_tensor = features_preprocessor.fit_transform(
    _embedded_train_features
)

_train_target_tensor = target_preprocessor.fit_transform(
    arabic_mnist_train_target
)

_train_dataset = utils.data.TensorDataset(
    _embedded_train_features_tensor,
    _train_target_tensor
)

embedded_train_dataloader = utils.data.DataLoader(
    _train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    persistent_workers=True,
    shuffle=True,
)

# create the validation dataset and dataloader
_embedded_val_features_tensor = features_preprocessor.transform(
    _embedded_val_features
)

_val_target_tensor = target_preprocessor.transform(
    arabic_mnist_val_target
)

_val_dataset = utils.data.TensorDataset(
    _embedded_val_features_tensor,
    _val_target_tensor
)

embedded_val_dataloader = utils.data.DataLoader(
    _val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False,
)

criterion = nn.CrossEntropyLoss()

# copy the weights from the model pretrained on MNIST data
unsupervised_pretraining_and_transfer_learning_model_fn = copy.deepcopy(pretrained_mnist_model_fn)
unsupervised_pretraining_and_transfer_learning_model_fn = unsupervised_pretraining_and_transfer_learning_model_fn.to(DEVICE)

optimizer = optim.SGD(
    unsupervised_pretraining_and_transfer_learning_model_fn.parameters(),
    lr=LEARNING_RATE
)

unsupervised_pretraining_and_transfer_learning_history_df = fit(
    criterion,
    unsupervised_pretraining_and_transfer_learning_model_fn,
    optimizer,
    embedded_train_dataloader,
    embedded_val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
unsupervised_pretraining_and_transfer_learning_accuracy = evaluate(
    unsupervised_pretraining_and_transfer_learning_model_fn,
    embedded_val_dataloader,
    accuracy_metric,
    device=DEVICE,
)

print(f"Unsupervised pretraining + Transfer learning accuracy: {unsupervised_pretraining_and_transfer_learning_accuracy.item(): .4f}")